# Логистическая регрессия: score, сигмоида и порог

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_bank_csv() -> Path:
    for path in (Path("bank_marketing_slim.csv"), Path("../../data/bank_marketing_slim.csv")):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_10_churn_logreg/data/bank_marketing_slim.csv"


CSV_PATH = find_bank_csv()
df = pd.read_csv(CSV_PATH)
target = df["y"].eq("yes").astype(int)
assert len(df) > 0 and set(target.unique()) == {0, 1}
assert "duration" in df.columns  # колонка видна только для разбора утечки
print(f"Строк: {len(df)}; доля yes: {target.mean():.3f}")


## 1. Момент предсказания

Разделите колонки на доступные до звонка и известные после него. `duration` должна оказаться только во второй группе.

In [ ]:
before_call = None  # TODO: все колонки, кроме y и duration
after_call = None   # TODO: список из duration
assert isinstance(before_call, list) and isinstance(after_call, list)
assert "duration" not in before_call
assert after_call == ["duration"]
assert set(before_call) | {"duration", "y"} == set(df.columns)


## 2. Почему `duration` — утечка

Опишите, в какой момент значение становится известно и почему его нельзя получить для ещё не совершённого звонка.

In [ ]:
LEAKAGE_NOTE = ""  # TODO: не менее 180 символов
assert len(LEAKAGE_NOTE) >= 180
assert "duration" in LEAKAGE_NOTE.lower()
assert any(word in LEAKAGE_NOTE.lower() for word in ["после", "звон"])


## 3. Линейный score

Реализуйте сумму `intercept + weight * value` для одного признака.

In [ ]:
def linear_score(value, weight, intercept):
    # TODO
    ...


assert abs(linear_score(2.0, 1.5, -1.0) - 2.0) < 1e-9
assert abs(linear_score(0.0, 7.0, -0.4) + 0.4) < 1e-9


## 4. Сигмоида

Преобразуйте число или массив score в значения строго между 0 и 1.

In [ ]:
def sigmoid(z):
    # TODO: используйте np.asarray и np.exp
    ...


grid = np.array([-4.0, 0.0, 4.0])
probabilities = sigmoid(grid)
assert probabilities.shape == grid.shape
assert np.all((probabilities > 0) & (probabilities < 1))
assert abs(float(sigmoid(0.0)) - 0.5) < 1e-9


## 5. Монотонность вероятности

Проверьте, как меняется вероятность на score от -6 до 6.

In [ ]:
score_grid = np.arange(-6.0, 7.0, 1.0)
probability_grid = None  # TODO
is_increasing = None     # TODO
assert len(probability_grid) == len(score_grid)
assert is_increasing is True
assert np.all(np.diff(probability_grid) > 0)


## 6. Функция порога

Верните бинарные метки: 1, если вероятность не меньше порога.

In [ ]:
def apply_threshold(proba, threshold=0.5):
    # TODO
    ...


demo = np.array([0.10, 0.30, 0.49, 0.50, 0.82])
assert apply_threshold(demo, 0.5).tolist() == [0, 0, 0, 1, 1]
assert apply_threshold(demo, 0.3).tolist() == [0, 1, 1, 1, 1]


## 7. Сколько клиентов попадёт в обзвон

Для пяти порогов посчитайте число положительных решений.

In [ ]:
demo_proba = np.array([0.06, 0.18, 0.24, 0.31, 0.47, 0.52, 0.68, 0.79, 0.91])
thresholds = [0.2, 0.35, 0.5, 0.65, 0.8]
selected_counts = None  # TODO: список чисел
assert len(selected_counts) == len(thresholds)
assert all(isinstance(value, (int, np.integer)) for value in selected_counts)
assert all(selected_counts[i] >= selected_counts[i + 1] for i in range(len(selected_counts) - 1))


## 8. Открытый эксперимент: порог под бюджет

Бюджет позволяет выбрать не более трёх клиентов. Найдите наименьший порог из сетки 0.05…0.95, который соблюдает бюджет.

In [ ]:
budget = 3
threshold_grid = np.arange(0.05, 1.0, 0.05)
budget_threshold = None  # TODO
chosen = None            # TODO: индексы выбранных клиентов
assert budget_threshold is not None
assert 0.05 <= budget_threshold <= 0.95
assert isinstance(chosen, np.ndarray) and len(chosen) <= budget


## 9. Самостоятельно: контракт решения

Напишите функцию, которая возвращает и метки, и число выбранных.

In [ ]:
def campaign_decision(proba, threshold):
    # TODO: return labels, selected_count
    ...


labels, count = campaign_decision(demo_proba, 0.5)
assert labels.shape == demo_proba.shape
assert set(np.unique(labels)) <= {0, 1}
assert count == int(labels.sum())
assert count == 4
